# 🚀 02. 문서 텍스트 추출, 청킹, pgvector 배치 적재 및 Parquet 백업

이 노트북은 PDF, DOCX, TXT 등 다양한 학술 문서 포맷에서 고속으로 텍스트를 추출하고, 오버랩 청킹을 수행해 LangChain Document 규격으로 표준화한 뒤 pgvector에 비동기 배치로 적재하고 Parquet 포맷으로 가공 백업하는 로직을 학습합니다.

### ⚙️ 필요한 라이브러리 및 모듈 임포트
텍스트 파싱을 위해 `pypdfium2`, `docx`, `xml` 모듈을 로드하고, 데이터 백업 시뮬레이션을 위해 `pandas`, `pyarrow` 패키지를 임포트합니다.

In [ ]:
import io
import os
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pypdfium2 as pdfium
from docx import Document as DocxDocument
from langchain_core.documents import Document
from langchain.embeddings import init_embeddings
from langchain_postgres import PGVector
from api.common.config import settings

## 📌 1단계. PDF 및 DOCX 고속 텍스트 추출 엔진 정의
ZIP 파일 포맷인 DOCX의 압축 파일 구조를 직접 파싱하는 XML 고속 추출 기법과 `pypdfium2`를 사용한 PDF 파싱 함수를 정의합니다.

In [ ]:
def extract_text_from_bytes(filename: str, file_bytes: bytes) -> str:
    fname = filename.lower()
    
    if fname.endswith(".pdf"):
        try:
            print(f"[PDF] Parsing {filename} using pypdfium2...")
            pdf = pdfium.PdfDocument(io.BytesIO(file_bytes))
            pages = []
            for page in pdf:
                textpage = page.get_textpage()
                pages.append(textpage.get_text_range() or "")
            return "\n".join(p for p in pages if p.strip())
        except Exception as e:
            print(f"PDF extraction failed: {e}")
            return ""
            
    elif fname.endswith(".docx") or fname.endswith(".doc"):
        try:
            print(f"[DOCX] Fast XML parsing: {filename}...")
            with zipfile.ZipFile(io.BytesIO(file_bytes)) as docx:
                xml_content = docx.read('word/document.xml')
            root = ET.fromstring(xml_content)
            namespaces = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
            texts = [node.text for node in root.findall('.//w:t', namespaces) if node.text]
            return "\n".join(texts)
        except Exception as e:
            print(f"Fast XML extraction failed, falling back to python-docx: {e}")
            try:
                doc = DocxDocument(io.BytesIO(file_bytes))
                return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
            except Exception as e2:
                print(f"DOCX extraction failed completely: {e2}")
                return ""
    else:
        try:
            return file_bytes.decode("utf-8")
        except UnicodeDecodeError:
            return file_bytes.decode("latin-1", errors="ignore")

## 📌 2단계. 텍스트 오버랩 청킹 및 4-키 표준 가공
추출된 본문 텍스트를 지정된 문자 개수 단위(예: 크기 800, 오버랩 150)로 조각내며, 각 청크가 독립된 컨텍스트를 가질 수 있도록 앞뒤 텍스트를 중첩시킵니다.

In [ ]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[str]:
    if not text.strip():
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap
    return chunks

# 가상 데이터 준비 및 텍스트 추출
mock_pdf = b"%PDF-1.4 ... Fake PDF bytes ... This is deep learning research text. We explain transformers."
extracted_text = extract_text_from_bytes("paper.pdf", mock_pdf)

if not extracted_text.strip():
    extracted_text = "This is fallback academic text. Attention mechanism is crucial. PostgreSQL pgvector index ensures fast search."
    print("Using fallback text due to mock format error.")
    
chunks = chunk_text(extracted_text, chunk_size=50, overlap=10)
print(f"Generated {len(chunks)} chunks:")
for idx, c in enumerate(chunks):
    print(f"  Chunk {idx+1}: {c}")

## 📌 3단계. pgvector 비동기 배치 적재
대용량 청크를 한 번에 데이터베이스에 넣을 때 발생하는 네트워크 부하와 시간 지연을 방지하기 위해 50건 단위 배치(`Batch`) 처리 기법을 실습합니다.

In [ ]:
# LangChain Document 변환 및 4-키 표준 메타데이터 주입
documents = []
for idx, chunk in enumerate(chunks):
    doc = Document(
        page_content=chunk,
        metadata={
            "arxiv_id": "2406.99999",
            "title": "A Mock Research Paper on Deep Learning",
            "domain": "cs",
            "source": "arxiv",
            "chunk_index": idx
        }
    )
    documents.append(doc)

# pgvector 배치 저장 실행
embeddings = init_embeddings(model="openai:text-embedding-3-large")
collection_name = "demo_batch_ingestion"
vectorstore = PGVector(
    embeddings=embeddings,
    collection_name=collection_name,
    connection=settings.PGVECTOR_URL,
    async_mode=True,
)

batch_size = 50
for i in range(0, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    await vectorstore.aadd_texts(
        texts=[d.page_content for d in batch],
        metadatas=[d.metadata for d in batch]
    )
    print(f"Ingested docs {i} to {min(i+batch_size, len(documents))}...")

# 데이터 확인 및 삭제
res = await vectorstore.asimilarity_search("attention mechanism", k=1)
print(f"Query test result: {res[0].page_content} | Metadata: {res[0].metadata}")
await vectorstore.adelete_collection()
print("Cleaned up demo batch collection.")

## 📌 4단계. Pandas/PyArrow 기반 Parquet 백업 및 복원
대용량 벡터 데이터셋의 백업과 버전 관리를 위해, 메모리 효율성과 스키마 무결성이 뛰어난 열 지향(Columnar) 데이터 구조인 **Parquet** 포맷으로 저장하고 로드하는 과정을 학습합니다.

In [ ]:
# Document 리스트를 Pandas DataFrame으로 가공
data_list = []
for d in documents:
    data_list.append({
        "page_content": d.page_content,
        "arxiv_id": d.metadata.get("arxiv_id"),
        "title": d.metadata.get("title"),
        "domain": d.metadata.get("domain"),
        "source": d.metadata.get("source"),
        "chunk_index": d.metadata.get("chunk_index")
    })

df = pd.DataFrame(data_list)
print("DataFrame Head:")
print(df.head(2))

backup_file = "temp_backup_demo.parquet"
print(f"\nSaving dataframe to Parquet format: {backup_file}...")
table = pa.Table.from_pandas(df)
pq.write_table(table, backup_file)

# 복원
print(f"Restoring dataframe from: {backup_file}...")
loaded_table = pq.read_table(backup_file)
loaded_df = loaded_table.to_pandas()
print(f"Successfully restored! DataFrame shape: {loaded_df.shape}")

# 임시 파일 삭제
if os.path.exists(backup_file):
    os.remove(backup_file)